In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


This notebook integrates the environmental impact and cost data derived in NB1 wiht the consumption data in the 2021 Scottish Health Survey

In [ ]:
%cd /content/drive/MyDrive/mSHIFT_SHeS/code_ocean/

/content/drive/MyDrive/mSHIFT_SHeS/code_ocean


In [ ]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
import sys

In [ ]:
# Add the parent directory of the notebook to sys.path, enables module imports from analysis_code
sys.path.append(str(Path().resolve() / 'code/notebooks/notebook_code'))

In [ ]:
from data_processing import add_env_data, add_sw

In [ ]:
data_path = Path("data")

In [ ]:
!pip install pyreadstat

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 30.9 MB/s eta 0:00:00


In [ ]:
import pyreadstat

In [ ]:
def load_data(path_to_data):

  if path_to_data.endswith('.csv'):
    data = pd.read_csv(path_to_data, low_memory=False, encoding = 'IBM819')
  elif path_to_data.endswith('.dta'):
    data, _ = pyreadstat.read_dta(path_to_data)

  return data

In [ ]:
####### SHeS 2021 ###########
diet_data_path = data_path / "shes_data_raw/SHeS_2021_food_level_condensed.csv"
ind_data_path = data_path / 'shes_data_raw/shes21i_eul.csv'

In [ ]:
ind_data = pd.read_csv(ind_data_path, low_memory=False, encoding = 'IBM819')
diet_data = pd.read_csv( diet_data_path, low_memory=False, encoding = 'IBM819')

In [ ]:
diet_data = pd.read_csv( diet_data_path, low_memory=False, encoding = 'IBM819')

In [ ]:
diet_data = pd.read_parquet(data_path / "diet_data.parquet")

In [ ]:
matched_items_data_path = data_path / "foodDB_data_processing/NDB_matching_data_FINAL.csv"
matched_items_data = pd.read_csv(matched_items_data_path)

In [ ]:
# Dataset derivied from NB1
df_impacts = pd.read_parquet(data_path / 'df_impacts_per_100g_1902.parquet')

In [ ]:
# Conversion factors that scale the impact and cost estiamtes to account for water diluation of some drink items
with open(data_path / "foodDB_data_processing/drink_conversion_factors.json", "r") as readfile:
    drink_conversion_factors = json.load(readfile)

In [ ]:
drink_conversion_factors

{'Tea': 0.015,
 'Coffee, instant': 0.089,
 'Coffee, fresh': 0.0515,
 'Decaf coffee, fresh': 0.0515,
 'Decaf coffee, instant': 0.089,
 'Herbal/ Fruit tea': 0.008,
 'Decaf tea': 0.015,
 'Green Tea': 0.015,
 'Hot / drinking chocolate, made with water, low calorie': 0.09,
 'Blackcurrant squash / juice, no added sugar, diluted': 0.152,
 'Orange squash, no added sugar, diluted': 0.25,
 'Fruit squash / juice, no added sugar, diluted': 0.25,
 'Orange squash / juice, with ADDED SUGAR, diluted': 0.25,
 'Orange squash, high juice, no added sugar, diluted': 0.25,
 'Ribena squash / juice, diluted, with ADDED SUGAR': 0.152,
 'Fruit squash / juice, with ADDED SUGAR, diluted': 0.25,
 'Ribena squash / juice , diluted, no added sugar': 0.152,
 'Apple and blackcurrant squash, no added sugar, diluted': 0.25,
 'Apple and blackcurrant squash / juice, with ADDED SUGAR, diluted': 0.25,
 'Blackcurrant squash / juice, with ADDED SUGAR, diluted': 0.152,
 'Fruit squash, high juice, no added sugar, diluted': 0.25,

In [ ]:
mean_env_columns = np.loadtxt(data_path / "indicator_lists/mean_env_columns.txt", dtype = str).tolist()
error_columns = np.loadtxt(data_path / "indicator_lists/error_columns.txt", dtype = str).tolist()
env_columns = np.loadtxt(data_path / "indicator_lists/env_columns.txt", dtype = str).tolist()

In [ ]:
# initialse the values for the new indicators
diet_data.loc[:, env_columns] = np.nan

In [ ]:
# add the foodDB indicators into the diet data
diet_data = diet_data.apply(lambda row: add_env_data(row=row,
                      NDB_data=matched_items_data,
                      df_impacts=df_impacts,
                      mean_env_columns=mean_env_columns,
                      error_columns = error_columns,
                      conversion_dict=drink_conversion_factors)
,
                            axis=1)

In [ ]:
# Add the individual level sample weight to hte diet data which is needed for the some the later analyses.
diet_data['Sample Weight'] = diet_data.apply(lambda row: add_sw(row, ind_data=ind_data), axis=1)

In [ ]:
# Save the diet data
diet_data.to_parquet(data_path / 'diet_data.parquet')